In [1]:
import pandas as pd

df = pd.read_csv('/Users/vedharai/hr-attrition-predictor/data/hr_clean.csv')
print(df.shape)

(1470, 45)


In [2]:
X = df.drop(columns=['Attrition'])
y = df['Attrition']
print(X.shape)
print(y.shape)

(1470, 44)
(1470,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, max_depth=3, min_samples_leaf=10)
rf_model.fit(X_train, y_train)
predictions = rf_model.predict(X_test)

In [22]:
from sklearn.metrics import classification_report
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.92      0.84      0.88       255
           1       0.33      0.51      0.40        39

    accuracy                           0.80       294
   macro avg       0.63      0.68      0.64       294
weighted avg       0.84      0.80      0.82       294



Random forest findings: Max attrition class recall obtained by manually tuning the model is 0.51. Logistic Regression is still beating the complex model with 0.59 class 1 recall. Dataset being highly imbalanced is the main reason for such low recall scores. 

In [26]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3,5,10],
    'min_samples_leaf':[5,10,15],
    'n_estimators':[100,200]
}

rf_base = RandomForestClassifier(class_weight='balanced', random_state=42)

grid_search = GridSearchCV(rf_base, param_grid, cv=5, scoring='recall')
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)

{'max_depth': 3, 'min_samples_leaf': 10, 'n_estimators': 100}
0.6160256410256411


In [28]:
print(classification_report(y_test, predictions))

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.              precision    recall  f1-score   support

           0       0.92      0.84      0.88       255
           1       0.33      0.51      0.40        39

    accuracy                           0.80       294
   macro avg       0.63      0.68      0.64       294
weighted avg       0.84      0.80      0.82       294




### Overall summary of ML models:

GridSearchCV found {'max_depth': 3, 'min_samples_leaf': 10, 'n_estimators': 100} to be the best combination of parameters. 
The recall of the model remained 0.51, same as the manually tuned Random Forest model.

| Model | Class 1 Recall |
| --- | --- |
| Logistic Regression | 0.59 |
| Random Forest (manual) | 0.51 |
| Random Forest (GridSearchCV) | 0.51 |

The model currently selected is the Logistic Regression model due to its high recall. 

The next step would be to try undersampling of the majority class to reduce imbalance.

In [30]:
from imblearn.under_sampling import RandomUnderSampler
undersampler = RandomUnderSampler(random_state=42)

X_resampled, y_resampled = undersampler.fit_resample(X_train, y_train)
print(y_resampled.value_counts())

Attrition
0    198
1    198
Name: count, dtype: int64


/Users/vedharai/hr-attrition-predictor/venv/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/vedharai/hr-attrition-predictor/venv/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


In [32]:
resampled_rf = RandomForestClassifier(max_depth=3, min_samples_leaf=10, n_estimators=100, class_weight='balanced', random_state=42)
resampled_rf.fit(X_resampled, y_resampled)
rs_predictions = resampled_rf.predict(X_test)
print(classification_report(y_test, rs_predictions))

              precision    recall  f1-score   support

           0       0.92      0.82      0.87       255
           1       0.32      0.56      0.41        39

    accuracy                           0.78       294
   macro avg       0.62      0.69      0.64       294
weighted avg       0.84      0.78      0.81       294



### Findings after resampling:

Resampling made a clear difference, it produced the highest random forest recall score so far. Updated table:


| Model | Class 1 Recall |
| --- | --- |
| Logistic Regression | 0.59 |
| Random Forest (manual) | 0.51 |
| Random Forest (GridSearchCV) | 0.51 |
| Random Forest (After undersampling) | 0.56 |